# Phân tích SQL – Dự đoán phí vận chuyển đơn hàng thương mại điện tử Olist
**Nhóm 8 · Lớp AI2107 · Môn ADY201M · FALL26**

---
## 1. Mục tiêu của bước SQL

Sau bước làm sạch (Phần 3), nhóm dùng SQL để **khám phá bảng dữ liệu sạch trước khi xây mô hình**. Bước này hỗ trợ các câu hỏi kinh doanh ở Phần 1:

- **Câu hỏi 1.** Trong các yếu tố như cân nặng, thể tích, khoảng cách giữa người bán và người mua, giá sản phẩm, yếu tố nào có mối liên hệ mạnh nhất với phí vận chuyển (`freight_value`)?
- **Câu hỏi 2.** Có thể dự đoán phí vận chuyển cho từng dòng sản phẩm với sai số trung bình bao nhiêu BRL, và các đặc điểm trên giải thích được bao nhiêu phần biến động của phí?
- **Câu hỏi 3.** Quan hệ giữa các đặc điểm trên và phí vận chuyển có mang tính tuyến tính hay không, tức là mô hình phi tuyến (Random Forest) có dự đoán tốt hơn mô hình tuyến tính (Linear Regression) hay không?

Notebook này gom **tất cả các truy vấn SQL đã dùng từ các bản trước đến bản cuối** (15 truy vấn). Bản chốt của nhóm gồm Query 1, 2, 2b, 3, 3b, 4 và 4c (đánh số lại 1–7 trong notebook 7 truy vấn); các truy vấn còn lại (4b, 5, 5b, 6, 6b, 7, 8) là bản mở rộng, chưa đưa vào báo cáo. SQL **không xếp hạng mức độ quan trọng** của các yếu tố vì các yếu tố liên quan với nhau (ví dụ cân nặng và thể tích); việc đó thuộc EDA (Phần 5) và mô hình (Phần 6). Câu hỏi 2 và Câu hỏi 3 được trả lời ở bước hồi quy (Phần 6) bằng MAE, RMSE, R² và so sánh hai mô hình.

## 2. Dữ liệu sử dụng

- **Nguồn:** Brazilian E-Commerce Public Dataset by Olist (Kaggle), giấy phép CC BY-NC-SA 4.0.
- **Bảng dùng trong SQL:** `olist_clean` (108.937 dòng, 12 cột) do Phần 3 xuất ra (tệp `olist_clean.csv`): chỉ giữ đơn đã giao, giai đoạn 01/2017–08/2018, bỏ dòng thiếu tọa độ, cân nặng hoặc kích thước, bỏ dòng có cân nặng bằng 0 và phí vận chuyển bằng 0.
- **Đơn vị quan sát:** một sản phẩm trong một đơn hàng, xác định bởi cặp (`order_id`, `order_item_id`).

| Cột | Ý nghĩa |
|---|---|
| `order_id`, `order_item_id` | Khóa của dòng (đơn hàng, số thứ tự sản phẩm trong đơn) |
| `seller_id` | Mã người bán |
| `order_purchase_timestamp` | Thời điểm đặt hàng (tháng đặt hàng được lấy từ cột này bằng `strftime` ở Query 7) |
| `price` | Giá bán của sản phẩm (BRL) |
| `weight_g` | Cân nặng sản phẩm (gam) |
| `volume_cm3` | Thể tích sản phẩm (dài × cao × rộng) |
| `distance_km` | Khoảng cách đường chim bay giữa người bán và khách (km), tính theo công thức Haversine từ tọa độ mã bưu chính |
| `category` | Danh mục sản phẩm (tiếng Anh; `unknown` nếu thiếu) |
| `order_n_items` | Số sản phẩm của đơn gốc (đếm trước khi loại dòng) |
| `same_state` | 1 nếu người bán và khách cùng bang, 0 nếu khác bang |
| `freight_value` | Phí vận chuyển (BRL), **biến cần dự đoán** |

## 3. Chuẩn bị môi trường
Nạp file dữ liệu sạch vào SQLite (cơ sở dữ liệu chạy trong bộ nhớ) và khai báo hàm `sql()` để chạy câu lệnh.

In [1]:
import sqlite3, pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Tệp do Phần 3 xuất ra. Colab: kéo olist_clean.csv vào /content/ rồi giữ nguyên dòng dưới.
# Trong repo: '../data/cleaned/olist_clean.csv'
CSV_PATH = '/content/olist_clean.csv'
df = pd.read_csv(CSV_PATH)

conn = sqlite3.connect(':memory:')
df.to_sql('olist_clean', conn, index=False, if_exists='replace')

def sql(query):
    """Chạy một câu SQL và trả về bảng kết quả"""
    return pd.read_sql_query(query, conn)

print(df.shape)

(108937, 12)


### Kiểm tra dữ liệu đã nạp
Xác nhận bảng nạp vào đúng với kết quả của bước làm sạch.

In [2]:
print('Số dòng, số cột        :', df.shape)
print('Tổng ô thiếu           :', df.isna().sum().sum())
print('Số dòng trùng khóa      :', df.duplicated(['order_id', 'order_item_id']).sum())
print('Giai đoạn đặt hàng      :', df['order_purchase_timestamp'].str[:7].min(), 'đến', df['order_purchase_timestamp'].str[:7].max())
print('Dòng category = unknown :', (df['category'] == 'unknown').sum())
assert df.shape == (108937, 12), 'Nạp nhầm tệp: cần bản 12 cột, 108.937 dòng'


Số dòng, số cột        : (108937, 12)
Tổng ô thiếu           : 0
Số dòng trùng khóa      : 0
Giai đoạn đặt hàng      : 2017-01 đến 2018-08
Dòng category = unknown : 1532


## 4. Các truy vấn

Cột `so_dong` là số dòng (sản phẩm trong đơn), không phải số đơn. Các truy vấn có `phi_TB` và `phi_trung_vi` tính trung vị bằng `ROW_NUMBER()`: phí lệch phải nên trung vị thấp hơn trung bình. Mốc chia nhóm là các giá trị tròn, dễ đọc, không chọn để tối ưu kết quả. Chia nhóm chỉ mô tả, không tách được tác động riêng của từng yếu tố.

---
## Query 1 - Tổng quan dữ liệu và phí vận chuyển

**Câu hỏi phân tích**: Bao nhiêu dòng, bao nhiêu đơn? Phí nhỏ nhất, trung bình, lớn nhất?

In [3]:
print("=== QUERY 1 ===")
print(sql("""
SELECT COUNT(*) AS so_dong,
       COUNT(DISTINCT order_id) AS so_don_duy_nhat,
       MIN(freight_value) AS phi_nho_nhat,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB,
       MAX(freight_value) AS phi_lon_nhat
FROM olist_clean;
"""))


=== QUERY 1 ===
   so_dong  so_don_duy_nhat  phi_nho_nhat  phi_ship_TB  phi_lon_nhat
0   108937            95377          0.01        20.01        409.68


**Nhận xét**: Dữ liệu có **108.937** dòng thuộc **95.377** đơn hàng. Phí trung bình là 20,01 BRL nhưng phí cao nhất lên tới 409,68 BRL (gấp khoảng **20 lần** mức trung bình), tức phí lệch phải mạnh. Vì vậy khi đánh giá mô hình cần báo MAE cùng sai số theo từng nhóm, không chỉ dựa vào mức trung bình.

---
## Query 2 - Phí theo nhóm cân nặng (trung bình và trung vị)

**Câu hỏi phân tích**: Hàng càng nặng thì phí có tăng không?

In [4]:
print("=== QUERY 2 ===")
print(sql("""
WITH b AS (
    SELECT freight_value, CASE WHEN weight_g < 500 THEN '1. Rất nhẹ (<500g)'
         WHEN weight_g < 2000 THEN '2. Nhẹ (500-2000g)'
         WHEN weight_g < 5000 THEN '3. Trung bình (2-5kg)'
         ELSE '4. Nặng (>5kg)' END AS nhom
    FROM olist_clean
),
r AS (
    SELECT nhom, freight_value,
           ROW_NUMBER() OVER (PARTITION BY nhom ORDER BY freight_value) AS rn,
           COUNT(*) OVER (PARTITION BY nhom) AS n
    FROM b
)
SELECT nhom,
       MAX(n) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_TB,
       ROUND(AVG(CASE WHEN rn IN ((n + 1) / 2, (n + 2) / 2) THEN freight_value END), 2) AS phi_trung_vi
FROM r
GROUP BY nhom
ORDER BY nhom;
"""))


=== QUERY 2 ===


                    nhom  so_dong  phi_TB  phi_trung_vi
0     1. Rất nhẹ (<500g)    43532   15.17         15.10
1     2. Nhẹ (500-2000g)    40632   17.89         16.33
2  3. Trung bình (2-5kg)    11220   22.81         20.02
3         4. Nặng (>5kg)    13553   39.62         29.90


**Nhận xét**: Phí trung bình tăng từ 15,17 BRL (dưới 500g) lên 39,62 BRL (trên 5kg), gấp **2,6 lần**; theo trung vị là 15,10 lên 29,90 BRL (gấp 2,0 lần). Trung bình của nhóm nặng cao hơn trung vị nhiều vì nhóm này có các phí rất lớn.

---
## Query 2b - Phí theo nhóm thể tích

**Câu hỏi phân tích**: Hàng càng cồng kềnh thì phí có tăng không?

In [5]:
print("=== QUERY 2B ===")
print(sql("""
WITH b AS (
    SELECT freight_value, CASE WHEN volume_cm3 < 2000 THEN '1. Nhỏ (<2000 cm3)'
         WHEN volume_cm3 < 5000 THEN '2. Vừa (2000-5000)'
         WHEN volume_cm3 < 15000 THEN '3. Lớn (5000-15000)'
         ELSE '4. Rất lớn (>15000)' END AS nhom
    FROM olist_clean
),
r AS (
    SELECT nhom, freight_value,
           ROW_NUMBER() OVER (PARTITION BY nhom ORDER BY freight_value) AS rn,
           COUNT(*) OVER (PARTITION BY nhom) AS n
    FROM b
)
SELECT nhom,
       MAX(n) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_TB,
       ROUND(AVG(CASE WHEN rn IN ((n + 1) / 2, (n + 2) / 2) THEN freight_value END), 2) AS phi_trung_vi
FROM r
GROUP BY nhom
ORDER BY nhom;
"""))


=== QUERY 2B ===


                  nhom  so_dong  phi_TB  phi_trung_vi
0   1. Nhỏ (<2000 cm3)    16927   14.91         14.73
1   2. Vừa (2000-5000)    30224   16.27         15.20
2  3. Lớn (5000-15000)    29070   18.61         16.39
3  4. Rất lớn (>15000)    32716   27.36         19.49


**Nhận xét**: Phí trung bình tăng từ 14,91 BRL lên 27,36 BRL (gấp **1,8 lần**); theo trung vị là 14,73 lên 19,49 BRL (gấp 1,3 lần). Mức tăng rõ nhất ở nhóm rất lớn. Thể tích và cân nặng liên quan với nhau nên không tách được tác động riêng.

---
## Query 3 - Phí theo nhóm khoảng cách

**Câu hỏi phân tích**: Khoảng cách người bán - người mua ảnh hưởng thế nào?

In [6]:
print("=== QUERY 3 ===")
print(sql("""
WITH b AS (
    SELECT freight_value, CASE WHEN distance_km < 200 THEN '1. Gần (<200km)'
         WHEN distance_km < 800 THEN '2. Trung bình (200-800km)'
         WHEN distance_km < 1500 THEN '3. Xa (800-1500km)'
         ELSE '4. Rất xa (>1500km)' END AS nhom
    FROM olist_clean
),
r AS (
    SELECT nhom, freight_value,
           ROW_NUMBER() OVER (PARTITION BY nhom ORDER BY freight_value) AS rn,
           COUNT(*) OVER (PARTITION BY nhom) AS n
    FROM b
)
SELECT nhom,
       MAX(n) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_TB,
       ROUND(AVG(CASE WHEN rn IN ((n + 1) / 2, (n + 2) / 2) THEN freight_value END), 2) AS phi_trung_vi
FROM r
GROUP BY nhom
ORDER BY nhom;
"""))


=== QUERY 3 ===


                        nhom  so_dong  phi_TB  phi_trung_vi
0            1. Gần (<200km)    28096   12.66         10.46
1  2. Trung bình (200-800km)    54032   19.80         16.39
2         3. Xa (800-1500km)    17099   23.74         18.66
3        4. Rất xa (>1500km)     9710   35.91         28.81


**Nhận xét**: Phí trung bình tăng từ 12,66 BRL (dưới 200km) lên 35,91 BRL (trên 1500km), gấp **2,8 lần**; theo trung vị là 10,46 lên 28,81 BRL. Mức tăng nhất quán giữa trung bình và trung vị.

---
## Query 3b - Phí theo nhóm giá bán

**Câu hỏi phân tích**: Giá cao thì phí có cao hơn không? (giá đi cùng hàng lớn, nặng nên chưa phải tác động riêng)

In [7]:
print("=== QUERY 3B ===")
print(sql("""
WITH b AS (
    SELECT freight_value, CASE WHEN price < 50 THEN '1. Rẻ (<50)'
         WHEN price < 100 THEN '2. 50-100'
         WHEN price < 200 THEN '3. 100-200'
         ELSE '4. Đắt (>200)' END AS nhom
    FROM olist_clean
),
r AS (
    SELECT nhom, freight_value,
           ROW_NUMBER() OVER (PARTITION BY nhom ORDER BY freight_value) AS rn,
           COUNT(*) OVER (PARTITION BY nhom) AS n
    FROM b
)
SELECT nhom,
       MAX(n) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_TB,
       ROUND(AVG(CASE WHEN rn IN ((n + 1) / 2, (n + 2) / 2) THEN freight_value END), 2) AS phi_trung_vi
FROM r
GROUP BY nhom
ORDER BY nhom;
"""))


=== QUERY 3B ===
            nhom  so_dong  phi_TB  phi_trung_vi
0    1. Rẻ (<50)    37965   14.73         14.52
1      2. 50-100    31941   18.01         16.19
2     3. 100-200    26093   23.13         18.55
3  4. Đắt (>200)    12938   34.16         22.84


**Nhận xét**: Phí trung bình tăng từ 14,73 BRL (giá dưới 50) lên 34,16 BRL (giá trên 200), gấp **2,3 lần**; theo trung vị là 14,52 lên 22,84 BRL (gấp 1,6 lần). Giá cao thường đi với hàng lớn và nặng nên chưa phải tác động riêng của giá.

---
## Query 4 - Cùng bang và khác bang

**Câu hỏi phân tích**: Giao cùng bang có rẻ hơn khác bang không?

In [8]:
print("=== QUERY 4 ===")
print(sql("""
WITH b AS (
    SELECT freight_value, CASE WHEN same_state = 1 THEN 'Cùng bang' ELSE 'Khác bang' END AS nhom
    FROM olist_clean
),
r AS (
    SELECT nhom, freight_value,
           ROW_NUMBER() OVER (PARTITION BY nhom ORDER BY freight_value) AS rn,
           COUNT(*) OVER (PARTITION BY nhom) AS n
    FROM b
)
SELECT nhom,
       MAX(n) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_TB,
       ROUND(AVG(CASE WHEN rn IN ((n + 1) / 2, (n + 2) / 2) THEN freight_value END), 2) AS phi_trung_vi
FROM r
GROUP BY nhom
ORDER BY nhom;
"""))
print(sql("""
-- Khoảng cách trung bình của hai nhóm
SELECT same_state, ROUND(AVG(distance_km), 0) AS khoang_cach_TB
FROM olist_clean
GROUP BY same_state;
"""))


=== QUERY 4 ===


        nhom  so_dong  phi_TB  phi_trung_vi
0  Cùng bang    39607   13.48         12.03
1  Khác bang    69330   23.74         18.26
   same_state  khoang_cach_TB
0           0           848.0
1           1           154.0


**Nhận xét**: Đơn khác bang có phí trung bình cao hơn **76%** (23,74 so với 13,48 BRL; trung vị 18,26 so với 12,03 BRL), nhưng khoảng cách trung bình cũng xa hơn nhiều (848km so với 154km). `same_state` liên quan chặt với khoảng cách.

---
## Query 4b - Cùng bang x khoảng cách

**Câu hỏi phân tích**: Trong cùng nhóm khoảng cách, khác bang có còn đắt hơn cùng bang không? (nhóm <100km khác bang chỉ 154 dòng)

In [9]:
print("=== QUERY 4B ===")
print(sql("""
WITH b AS (
    SELECT freight_value, same_state,
           CASE WHEN distance_km < 100 THEN '1. <100km'
                WHEN distance_km < 200 THEN '2. 100-200km'
                WHEN distance_km < 400 THEN '3. 200-400km'
                ELSE '4. >400km' END AS nhom_khoang_cach
    FROM olist_clean
)
SELECT nhom_khoang_cach, same_state, COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_TB
FROM b
GROUP BY nhom_khoang_cach, same_state
ORDER BY nhom_khoang_cach, same_state;
"""))


=== QUERY 4B ===
  nhom_khoang_cach  same_state  so_dong  phi_TB
0        1. <100km           0      154   16.09
1        1. <100km           1    20254   11.72
2     2. 100-200km           0     1201   18.13
3     2. 100-200km           1     6487   14.51
4     3. 200-400km           0    13329   21.11
5     3. 200-400km           1     9674   15.57
6        4. >400km           0    54646   24.53
7        4. >400km           1     3192   16.18


**Nhận xét**: Trong cùng nhóm khoảng cách, khác bang vẫn đắt hơn cùng bang: 18,13 so với 14,51 BRL (100–200km), 21,11 so với 15,57 (200–400km), 24,53 so với 16,18 (trên 400km). Nhóm dưới 100km khác bang chỉ có 154 dòng nên không diễn giải. Đây chỉ là mô tả, chưa phải nhân quả.

---
## Query 4c - Cân nặng x khoảng cách

**Câu hỏi phân tích**: Khoảng cách làm phí tăng như nhau ở hàng nhẹ và hàng nặng không?

In [10]:
print("=== QUERY 4C ===")
print(sql("""
WITH b AS (
    SELECT freight_value,
           CASE WHEN weight_g < 500 THEN '1. <500g'
                WHEN weight_g < 2000 THEN '2. 500-2000g'
                WHEN weight_g < 5000 THEN '3. 2-5kg'
                ELSE '4. >5kg' END AS nhom_can_nang,
           CASE WHEN distance_km < 200 THEN '1. <200km'
                WHEN distance_km < 800 THEN '2. 200-800km'
                WHEN distance_km < 1500 THEN '3. 800-1500km'
                ELSE '4. >1500km' END AS nhom_khoang_cach
    FROM olist_clean
)
SELECT nhom_can_nang, nhom_khoang_cach, COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_TB
FROM b
GROUP BY nhom_can_nang, nhom_khoang_cach
ORDER BY nhom_can_nang, nhom_khoang_cach;
"""))


=== QUERY 4C ===


   nhom_can_nang nhom_khoang_cach  so_dong  phi_TB
0       1. <500g        1. <200km    12673    9.17
1       1. <500g     2. 200-800km    20069   15.39
2       1. <500g    3. 800-1500km     6898   18.32
3       1. <500g       4. >1500km     3892   27.95
4   2. 500-2000g        1. <200km     9253   11.75
5   2. 500-2000g     2. 200-800km    21418   17.15
6   2. 500-2000g    3. 800-1500km     6305   20.93
7   2. 500-2000g       4. >1500km     3656   32.49
8       3. 2-5kg        1. <200km     2706   15.14
9       3. 2-5kg     2. 200-800km     5615   22.04
10      3. 2-5kg    3. 800-1500km     1877   26.86
11      3. 2-5kg       4. >1500km     1022   39.94
12       4. >5kg        1. <200km     3464   25.93
13       4. >5kg     2. 200-800km     6930   38.92
14       4. >5kg    3. 800-1500km     2019   48.10
15       4. >5kg       4. >1500km     1140   70.47


**Nhận xét**: Từ gần đến rất xa, hàng dưới 500g tăng từ 9,17 lên 27,95 BRL (**+18,8 BRL**, gấp 3,0 lần), hàng trên 5kg tăng từ 25,93 lên 70,47 BRL (**+44,5 BRL**, gấp 2,7 lần). Mức tăng tuyệt đối lớn hơn ở hàng nặng nên cân nặng và khoảng cách có tương tác, không cộng đơn giản. Ô nhỏ nhất có 1.022 dòng.

---
## Query 5 - Top 10 danh mục phí trung bình cao nhất (bản gốc)

**Câu hỏi phân tích**: Danh mục nào đắt nhất, có liên hệ gì với cân nặng? (chỉ xét danh mục từ 300 dòng)

In [11]:
print("=== QUERY 5 ===")
print(sql("""
SELECT category,
       COUNT(*) AS so_dong,
       ROUND(AVG(weight_g), 0) AS trong_luong_TB,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM olist_clean
GROUP BY category
HAVING COUNT(*) >= 300
ORDER BY phi_ship_TB DESC
LIMIT 10;
"""))


=== QUERY 5 ===


                          category  so_dong  trong_luong_TB  phi_ship_TB
0                 office_furniture     1655         11329.0        40.17
1            furniture_living_room      443          8727.0        37.43
2              luggage_accessories     1073          5788.0        27.83
3              musical_instruments      650          3135.0        27.46
4        construction_tools_lights      300          3358.0        24.80
5                 small_appliances      658          2929.0        23.33
6                     garden_tools     4196          2827.0        22.94
7                home_construction      596          3037.0        22.84
8                             baby     2961          3271.0        22.24
9  construction_tools_construction      909          3080.0        22.08


**Nhận xét**: Đắt nhất là đồ nội thất văn phòng (40,17 BRL) và nội thất phòng khách (37,43 BRL), đồng thời cũng là hai nhóm nặng nhất trong top 10 (khoảng 11,3kg và 8,7kg). Vì danh mục và cân nặng liên quan chặt với nhau, bảng này không tách được tác động riêng của danh mục; cần giữ cả hai trong mô hình và kiểm tra vai trò bổ sung của danh mục ở bước hồi quy.

---
## Query 5b - Top 10 danh mục, có phí mỗi kg và chênh so với trung bình chung (dùng JOIN)

**Câu hỏi phân tích**: Như Query 5, thêm phí mỗi kg để thấy phí không tỷ lệ thẳng với cân nặng

In [12]:
print("=== QUERY 5B ===")
print(sql("""
WITH tong AS (SELECT AVG(freight_value) AS tb_chung FROM olist_clean),
c AS (
    SELECT category,
           COUNT(*) AS so_dong,
           ROUND(AVG(weight_g) / 1000.0, 2) AS kg_TB,
           ROUND(AVG(freight_value), 2) AS phi_TB,
           ROUND(SUM(freight_value) / (SUM(weight_g) / 1000.0), 2) AS phi_moi_kg
    FROM olist_clean
    GROUP BY category
    HAVING COUNT(*) >= 300
)
SELECT c.*, ROUND(c.phi_TB - t.tb_chung, 2) AS chenh_so_voi_TB_chung
FROM c JOIN tong t
ORDER BY c.phi_TB DESC
LIMIT 10;
"""))


=== QUERY 5B ===


                          category  so_dong  kg_TB  phi_TB  phi_moi_kg  chenh_so_voi_TB_chung
0                 office_furniture     1655  11.33   40.17        3.55                  20.16
1            furniture_living_room      443   8.73   37.43        4.29                  17.42
2              luggage_accessories     1073   5.79   27.83        4.81                   7.82
3              musical_instruments      650   3.13   27.46        8.76                   7.45
4        construction_tools_lights      300   3.36   24.80        7.39                   4.79
5                 small_appliances      658   2.93   23.33        7.97                   3.32
6                     garden_tools     4196   2.83   22.94        8.11                   2.93
7                home_construction      596   3.04   22.84        7.52                   2.83
8                             baby     2961   3.27   22.24        6.80                   2.23
9  construction_tools_construction      909   3.08   22.08  

**Nhận xét**: Hai nhóm nặng nhất (nội thất văn phòng khoảng 11,3kg và phòng khách khoảng 8,7kg) có phí mỗi kg thấp hơn (3,55 và 4,29 BRL/kg) so với nhóm khoảng 3kg (6,8 đến 8,8 BRL/kg), nên phí không tỷ lệ thẳng với cân nặng. Danh mục và cân nặng liên quan chặt nên không tách được tác động riêng của danh mục.

---
## Query 6 - Phí theo số sản phẩm trong đơn

**Câu hỏi phân tích**: Đơn nhiều sản phẩm có phí mỗi dòng khác đơn một sản phẩm không?

In [13]:
print("=== QUERY 6 ===")
print(sql("""
WITH b AS (
    SELECT freight_value,
           CASE WHEN order_n_items = 1 THEN '1 sản phẩm' ELSE '>=2 sản phẩm' END AS nhom
    FROM olist_clean
)
SELECT nhom, COUNT(*) AS so_dong, ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM b
GROUP BY nhom;
"""))


=== QUERY 6 ===
           nhom  so_dong  phi_ship_TB
0    1 sản phẩm    85856        20.39
1  >=2 sản phẩm    23081        18.59


**Nhận xét**: Đơn có từ 2 sản phẩm trở lên có phí trung bình mỗi dòng thấp hơn (18,59 BRL so với 20,39 BRL, thấp hơn khoảng 1,80 BRL). Hai nhóm này có thể khác nhau về cân nặng và khoảng cách, nên Query 6b so sánh lại trong cùng nhóm cân nặng và khoảng cách.

---
## Query 6b - Đơn 1 sản phẩm và nhiều sản phẩm, cùng nhóm cân nặng và khoảng cách

**Câu hỏi phân tích**: Chênh lệch ở Query 6 có còn khi giữ cố định cân nặng và khoảng cách không?

In [14]:
print("=== QUERY 6B ===")
print(sql("""
WITH b AS (
    SELECT freight_value, order_n_items,
           CASE WHEN weight_g < 2000 THEN '1. Nhẹ (<2kg)'
                WHEN weight_g < 5000 THEN '2. Trung bình (2-5kg)'
                ELSE '3. Nặng (>5kg)' END AS nhom_trong_luong,
           CASE WHEN distance_km < 200 THEN '1. Gần (<200km)'
                WHEN distance_km < 800 THEN '2. Trung bình (200-800km)'
                ELSE '3. Xa (>800km)' END AS nhom_khoang_cach
    FROM olist_clean
)
SELECT nhom_trong_luong, nhom_khoang_cach,
       SUM(order_n_items = 1) AS so_dong_1_sp,
       SUM(order_n_items >= 2) AS so_dong_nhieu_sp,
       ROUND(AVG(CASE WHEN order_n_items = 1 THEN freight_value END), 2) AS phi_TB_1_sp,
       ROUND(AVG(CASE WHEN order_n_items >= 2 THEN freight_value END), 2) AS phi_TB_nhieu_sp
FROM b
GROUP BY nhom_trong_luong, nhom_khoang_cach
ORDER BY nhom_trong_luong, nhom_khoang_cach;
"""))


=== QUERY 6B ===


        nhom_trong_luong           nhom_khoang_cach  so_dong_1_sp  so_dong_nhieu_sp  phi_TB_1_sp  phi_TB_nhieu_sp
0          1. Nhẹ (<2kg)            1. Gần (<200km)         17207              4719        10.32            10.03
1          1. Nhẹ (<2kg)  2. Trung bình (200-800km)         31979              9508        16.44            15.82
2          1. Nhẹ (<2kg)             3. Xa (>800km)         16678              4073        23.85            21.64
3  2. Trung bình (2-5kg)            1. Gần (<200km)          2151               555        15.34            14.36
4  2. Trung bình (2-5kg)  2. Trung bình (200-800km)          4573              1042        22.31            20.85
5  2. Trung bình (2-5kg)             3. Xa (>800km)          2423               476        32.06            28.51
6         3. Nặng (>5kg)            1. Gần (<200km)          2750               714        26.29            24.54
7         3. Nặng (>5kg)  2. Trung bình (200-800km)          5555              1375     

**Nhận xét**: Trong cả 9 nhóm, phí trung bình mỗi dòng của đơn nhiều sản phẩm đều thấp hơn đơn một sản phẩm, thấp hơn từ khoảng 0,29 đến 4,66 BRL; ô nhỏ nhất vẫn có 476 dòng. Điều này gợi ý có ưu đãi hoặc cách chia phí khi gộp đơn, nhưng cũng có thể do khác biệt khác (loại sản phẩm, người bán). Phần 3 (mục 9.2) cũng chưa xác định được cách tính phí của đơn nhiều sản phẩm, nên chưa kết luận; sai số của nhóm này sẽ được báo riêng ở bước hồi quy.

---
## Query 7 - Phí trung bình theo tháng

**Câu hỏi phân tích**: Phí có thay đổi theo mùa vụ không?

In [15]:
print("=== QUERY 7 ===")
print(sql("""
SELECT strftime('%Y-%m', order_purchase_timestamp) AS month,
       COUNT(*) AS so_dong,
       ROUND(AVG(freight_value), 2) AS phi_ship_TB
FROM olist_clean
GROUP BY month
ORDER BY month;
"""))


=== QUERY 7 ===


      month  so_dong  phi_ship_TB
0   2017-01      912        17.17
1   2017-02     1858        19.92
2   2017-03     2883        19.00
3   2017-04     2548        19.53
4   2017-05     3983        19.34
5   2017-06     3459        19.53
6   2017-07     4393        19.15
7   2017-08     4756        19.03
8   2017-09     4711        19.77
9   2017-10     5178        19.74
10  2017-11     8435        19.52
11  2017-12     6152        18.92
12  2018-01     7998        19.07
13  2018-02     7478        18.58
14  2018-03     7981        20.86
15  2018-04     7634        20.74
16  2018-05     7558        19.91
17  2018-06     6981        22.23
18  2018-07     6934        22.95
19  2018-08     7105        20.57


**Nhận xét**: Phần lớn các tháng phí dao động quanh 19–20 BRL; **không thấy cao điểm cuối năm** (tháng 11 và 12/2017: 19,52 và 18,92 BRL). Tháng 6 và 7/2018 cao hơn (22,23 và 22,95 BRL) nhưng không có xu hướng tăng đều. Yếu tố thời gian ảnh hưởng nhẹ hơn nhiều so với khoảng cách và trọng lượng; dữ liệu chỉ có 20 tháng (các tháng 1–8 xuất hiện 2 lần, tháng 9–12 chỉ 1 lần) nên chưa đủ để kết luận chắc về mùa vụ. Nên chia dữ liệu theo thời gian (các tháng cuối để kiểm tra) khi huấn luyện.

---
## Query 8 - Top 10 dòng có phí cao nhất

**Câu hỏi phân tích**: Các dòng phí cao nhất có phải lỗi dữ liệu không?

In [16]:
print("=== QUERY 8 ===")
print(sql("""
SELECT order_id, category, weight_g, volume_cm3,
       ROUND(distance_km, 0) AS khoang_cach_km,
       price, order_n_items, freight_value
FROM olist_clean
ORDER BY freight_value DESC
LIMIT 10;
"""))


=== QUERY 8 ===


                           order_id                         category  weight_g  volume_cm3  khoang_cach_km    price  order_n_items  freight_value
0  a77e1550db865202c56b19ddc6dc4d53                             baby   14675.0    183040.0          1865.0   979.00              1         409.68
1  076d1555fb53a89b0ef4d529e527a0f6                       housewares   30000.0    282750.0           457.0  2338.08              1         375.28
2  3fde74c28a3d5d618c00f26d51baafa0                       housewares   30000.0    282750.0           450.0  2338.08              1         375.28
3  9f49bd16053df810384e793386312674                       housewares   25250.0    296208.0           617.0  1149.00              1         339.59
4  264a7e199467906c0727394df82d1a6a                    health_beauty   30000.0    251625.0          1708.0  1050.00              1         338.30
5  c7a07ddd52bbe18b61da49a8d89853d3                    health_beauty   30000.0    251625.0           433.0  1050.00         

**Nhận xét**: Cả 10 dòng đều là hàng nặng, phí 314 đến 410 BRL, phù hợp với hàng nặng hoặc đi xa và không có dấu hiệu lỗi nhập liệu; nhất quán với quyết định giữ giá trị ngoại lai ở Phần 3 (mục 9.7). Hai dòng có cùng cân nặng, giá và phí nhưng khác mã đơn, có thể là hai đơn giống nhau.

---
## 5. Tổng hợp phát hiện

| Query | Phát hiện |
|---|---|
| 1 | Phí trung bình 20,01 BRL, cao nhất 409,68 BRL (gấp khoảng 20 lần) |
| 2, 2b, 3, 3b | Phí tăng theo cân nặng (15,17 → 39,62), thể tích (14,91 → 27,36), khoảng cách (12,66 → 35,91), giá (14,73 → 34,16) |
| 4, 4b | Cùng bang 13,48, khác bang 23,74; trong cùng nhóm khoảng cách khác bang vẫn đắt hơn nhưng chỉ là mô tả |
| 4c | Khoảng cách làm phí tăng +18,8 BRL (hàng dưới 500g) và +44,5 BRL (hàng trên 5kg): có tương tác |
| 5, 5b | Nội thất văn phòng, phòng khách đắt nhất và nặng nhất; phí mỗi kg thấp hơn ở nhóm nặng |
| 6, 6b | Đơn từ 2 sản phẩm có phí mỗi dòng thấp hơn (18,59 so với 20,39), thấp hơn ở cả 9 nhóm |
| 7 | Phí quanh 19–20 BRL phần lớn các tháng, không có cao điểm cuối năm |
| 8 | 10 dòng phí cao nhất đều là hàng nặng, không có dấu hiệu lỗi |

## 6. Hạn chế và bước tiếp theo

**Hạn chế:**
- Chỉ so sánh theo nhóm, không tách được tác động riêng của từng yếu tố khi các yếu tố liên quan với nhau; kết quả chỉ mô tả, chưa phải kết luận về nguyên nhân hay mức độ quan trọng.
- Chưa xác định được phí của đơn nhiều sản phẩm là phí riêng hay phí của cả đơn được chia (Phần 3, mục 9.2).
- Khoảng cách tính từ tọa độ mã bưu chính theo đường chim bay, không phải quãng đường thực.
- Dữ liệu không có thông tin hãng vận chuyển, giá cước hay giá nhiên liệu.

**Bước tiếp theo:** EDA bằng Python (Phần 5) rồi hồi quy (Phần 6: Linear Regression và Random Forest, đo bằng MAE, RMSE, R²).

In [17]:
conn.close()
print('\nĐã đóng kết nối SQLite.')



Đã đóng kết nối SQLite.
